# Domain 2: Output Evaluation and Validation
**Exam**: Claude Certified Associate – Foundations (CCAO-F)
**Weighting**: 21% of exam
**Depth**: deep

## Domain Overview

This is the largest domain on the exam. It covers critically evaluating Claude's output rather
than accepting it at face value: spotting hallucinations and inconsistencies, fact-checking claims,
deciding when a human needs to review something, and adapting/organizing output for its intended
audience and format.

**Why it matters**: the exam's single largest domain reflects that the most common real-world
failure mode for AI-assisted work is trusting unverified output. Associates are expected to be the
last line of defense before content reaches stakeholders.

**Key concepts to know**: hallucination detection, fact-checking workflows, escalation triggers,
audience-appropriate editing, and output-format selection (artifacts vs. inline vs. structured data).

In [ ]:
# Setup: a small helper so every cell in this notebook runs end-to-end,
# whether or not you have an ANTHROPIC_API_KEY configured.
import os

def call_claude(prompt, system=None, model="claude-sonnet-4-5", max_tokens=600):
    """Calls the real Claude API if ANTHROPIC_API_KEY is set; otherwise returns a
    simulated response so this notebook is runnable without API access."""
    try:
        import anthropic
        api_key = os.environ.get("ANTHROPIC_API_KEY")
        if not api_key:
            raise RuntimeError("no ANTHROPIC_API_KEY set")
        client = anthropic.Anthropic(api_key=api_key)
        kwargs = {"model": model, "max_tokens": max_tokens, "messages": [{"role": "user", "content": prompt}]}
        if system:
            kwargs["system"] = system
        resp = client.messages.create(**kwargs)
        return resp.content[0].text
    except Exception as e:
        return (f"[SIMULATED RESPONSE - set ANTHROPIC_API_KEY and `pip install anthropic` "
                f"to call the real API]\nPrompt received ({len(prompt)} chars): {prompt[:150]}...")

print("call_claude() ready. Set ANTHROPIC_API_KEY as an environment variable to hit the real API.")

## Key Concepts

### Evaluate Claude-generated outputs for accuracy and completeness

Accuracy means claims are true; completeness means nothing material is missing. A good
evaluation checklist asks: does every factual claim have a plausible, checkable source? Does the
output address every part of the original request?

In [ ]:
request = "Summarize our refund policy: 30-day window, store credit only after day 14, no refunds on final-sale items."
output = call_claude(f"Write a customer-facing summary of this policy: {request}")

checklist = {
    "mentions 30-day window": "30" in output or "thirty" in output.lower(),
    "mentions store credit after day 14": "credit" in output.lower(),
    "mentions final-sale exception": "final" in output.lower() or "sale" in output.lower(),
}
print(output)
print(checklist)

### Identify hallucinations, inconsistencies, and biases in responses

A hallucination is a confidently stated but fabricated detail (a fake statistic, a made-up
citation, a nonexistent policy clause). Inconsistencies are internal contradictions. Bias shows up as
skewed framing that favors one interpretation without justification.

In [ ]:
suspicious_output = call_claude(
    "Summarize the key findings of the (fictional) 2024 'Global Workplace AI Adoption Survey', citing the exact percentage of adoption."
)
print(suspicious_output)

print("""
Review checklist:
- Does this cite a specific number for a survey that was never described to Claude with real data? -> likely hallucinated
- Would you be able to find this exact source and number online? If not, treat as unverified.
""")

### Apply fact-checking and validation techniques

Validation techniques include: asking Claude to cite its reasoning or flag uncertainty, cross-
checking numeric claims against your own source data, and never forwarding an unverified specific
claim (dates, statistics, legal/compliance details) without independent confirmation.

In [ ]:
claim = call_claude("What is our current customer NPS score?")
print("Claude said:", claim)

source_of_truth = {"nps": 42}  # your actual internal data
print("Validation: does the internal record match the claim above?",
      str(source_of_truth["nps"]) in claim)

### Determine when human review or additional verification is required

High-stakes categories (legal, financial, medical, compliance, anything sent externally at
scale) always warrant human review before publishing, regardless of how confident the output
sounds. Low-stakes, easily-verified categories may not.

In [ ]:
def needs_human_review(category: str) -> bool:
    high_stakes = {"legal", "financial", "medical", "compliance", "external-press"}
    return category.lower() in high_stakes

for cat in ["legal", "internal-brainstorm", "financial", "meeting-notes"]:
    print(cat, "-> human review required:", needs_human_review(cat))

### Edit, adapt, refine, and compare outputs for the intended audience

The same content needs different framing for executives (concise, outcome-focused),
customers (benefit-focused, plain language), and engineers (precise, technical). Adapting is a
distinct skill from drafting from scratch.

In [ ]:
base_content = "We migrated our database to reduce query latency by 40% and cut infra cost by $12k/month."

for audience in ["executive", "customer", "engineer"]:
    adapted = call_claude(f"Rewrite this update for a {audience} audience: {base_content}")
    print(f"[{audience}]", adapted[:150], "...\n")

### Organize and curate information and select appropriate output formats

Claude.ai can return inline prose, a structured Artifact (document, table, code), or
structured data (JSON/CSV-like). Choose the format based on how the output will be used
downstream (read once vs. reused vs. imported into another tool).

In [ ]:
use_cases = {
    "one-time executive readout": "inline prose",
    "reusable onboarding checklist": "artifact (document)",
    "data to import into a spreadsheet": "structured data (CSV/table)",
    "a small script to run once": "artifact (code)",
}
for use_case, best_format in use_cases.items():
    print(f"{use_case:40s} -> {best_format}")

## Mini Hands-On Project

**Covers**: all Domain 2 sub-objectives.

**Scenario**: Claude has drafted a customer-facing incident report. Before it goes to the status
page, you must evaluate it for accuracy, flag anything needing human review, adapt it for a
non-technical audience, and pick the right output format.

### Step 1: Generate the draft to evaluate

In [ ]:
incident_facts = "Outage from 14:02-14:47 UTC affected 8% of API requests; root cause was a bad config push; rolled back at 14:41."
draft_report = call_claude(f"Write a public incident report based on: {incident_facts}")
print(draft_report)

### Step 2: Evaluate for accuracy/hallucination against the source facts

In [ ]:
facts_present = {
    "start time (14:02)": "14:02" in draft_report,
    "end time (14:47)": "14:47" in draft_report,
    "8% impact figure": "8%" in draft_report,
    "root cause (config push)": "config" in draft_report.lower(),
}
print(facts_present)
print("Any fact missing or a number present that isn't in the source? Flag it before publishing.")

### Step 3: Decide on human review and adapt for the audience

In [ ]:
category = "external-press"  # public status pages count as external communication
print("Needs human review:", category in {"legal", "financial", "medical", "compliance", "external-press"})

customer_version = call_claude(f"Rewrite this incident report for a non-technical customer audience, reassuring tone: {draft_report}")
print(customer_version)

## Lab Exercises

### Exercise 1: Hallucination hunt

**Objective**: Identify fabricated specifics in a Claude-generated summary.

**Produce**: A list of flagged claims with reasoning for each.

In [ ]:
summary = call_claude("Summarize the (fictional) internal report 'Project Nightingale Q2 Results' in 3 bullet points with exact figures.")
print(summary)

# TODO: list which claims in `summary` are unverifiable/likely hallucinated and why
flagged_claims = [
    # ("claim text", "reason it's suspicious"),
]
for claim, reason in flagged_claims:
    print(f"FLAGGED: {claim} -- {reason}")

# Expected output: At least 2 flagged claims (e.g., specific numbers or a report that was never provided as real source data).

In [ ]:
# STRETCH GOAL: write a simple heuristic function that flags any sentence containing a
# percentage or dollar figure as "needs verification" for review purposes.
import re
def flag_numeric_claims(text: str):
    # TODO: return list of sentences containing a number or % or $
    pass
print(flag_numeric_claims(summary))

# Expected output: A list of sentences from the summary containing numeric or currency figures, flagged for verification.

### Exercise 2: Fact-check against source data

**Objective**: Cross-check a Claude claim against a provided source-of-truth dictionary.

**Produce**: A pass/fail validation result and a corrected version if it fails.

In [ ]:
source_of_truth = {"active_users": 18400, "churn_rate": "2.8%"}
claim = call_claude("What is our current churn rate?")
print(claim)

# TODO: check whether source_of_truth["churn_rate"] appears in `claim`; if not, generate a corrected statement
matches = source_of_truth["churn_rate"] in claim
corrected = claim if matches else f"Corrected: churn rate is {source_of_truth['churn_rate']}."
print(matches, corrected)

# Expected output: A boolean validation result plus a corrected statement when the claim doesn't match the source.

In [ ]:
# STRETCH GOAL: generalize to validate every field in source_of_truth against a longer report.
report = call_claude("Write a 3-sentence status update covering active users and churn rate.")
def validate_report(report: str, facts: dict):
    # TODO: return a dict of {fact_name: found_in_report_bool}
    pass
print(validate_report(report, source_of_truth))

# Expected output: A generalized validator that checks multiple fields from source_of_truth against a block of text at once.

### Exercise 3: Escalation decision log

**Objective**: Build a small decision log classifying 5 outputs as auto-publish vs. needs-review.

**Produce**: A table/list of 5 content categories with pass/fail human-review decisions and justification.

In [ ]:
items = [
    ("internal FAQ answer", "internal"),
    ("press release quote", "external-press"),
    ("HR policy summary", "compliance"),
    ("brainstorm list for a team offsite", "internal"),
    ("customer contract clause explanation", "legal"),
]

# TODO: for each item, decide if it needs human review and write a one-line justification
for name, category in items:
    needs_review = None  # TODO
    justification = ""  # TODO
    print(name, "->", needs_review, "--", justification)

# Expected output: 5 correctly justified review decisions, with legal/compliance/external-press flagged true.

In [ ]:
# STRETCH GOAL: extend the decision log with a required reviewer role per category
# (e.g., legal counsel, comms lead, engineering lead).
reviewer_map = {}  # TODO: map category -> required reviewer role
for name, category in items:
    print(name, "-> reviewer:", reviewer_map.get(category, "none required"))

# Expected output: Each high-stakes category mapped to an appropriate named reviewer role.

### Exercise 4: Audience adaptation triple

**Objective**: Adapt one technical update for 3 different audiences and compare.

**Produce**: Three adapted versions plus a short note on what changed each time.

In [ ]:
technical_update = "We added rate limiting (100 req/min per key) to the public API to prevent abuse after a traffic spike."

# TODO: adapt technical_update for "customer", "executive", and "support-team" audiences
audiences = ["customer", "executive", "support-team"]
adapted = {}
for a in audiences:
    adapted[a] = call_claude("...")  # TODO
    print(a, "->", adapted[a][:120], "...")

# Expected output: Three distinctly-toned adaptations of the same underlying fact.

In [ ]:
# STRETCH GOAL: ask Claude to explain, in one sentence per audience, what it changed and why.
for a in audiences:
    explanation = call_claude(f"In one sentence, explain what tone/content changes you made for the {a} audience version: {adapted.get(a, '')}")
    print(a, "explanation:", explanation)

# Expected output: A short, distinct rationale per audience explaining the adaptation choices.

### Exercise 5: Format selection exercise

**Objective**: Given 4 output scenarios, choose and justify the best output format.

**Produce**: A mapping of scenario -> chosen format -> justification.

In [ ]:
scenarios = [
    "A 40-row list of leads to hand off to sales for CRM import",
    "A one-time answer to 'what should I say in this email'",
    "A reusable runbook for handling server outages",
    "A small Python script to rename a batch of files",
]

# TODO: for each scenario choose a format (inline prose / artifact-document / artifact-code / structured data)
# and justify the choice in one sentence
for s in scenarios:
    fmt = "..."  # TODO
    justification = "..."  # TODO
    print(s, "->", fmt, "--", justification)

# Expected output: 4 scenario -> format mappings with sound one-sentence justifications.

In [ ]:
# STRETCH GOAL: actually generate the CRM-import list as structured data (list of dicts) using call_claude,
# then verify the structure is consistent (same keys) for every row.
leads_prompt = "Generate 5 fictional sales leads as a list of dicts with keys: name, company, email."
leads_response = call_claude(leads_prompt)
print(leads_response)

# Expected output: Output resembling structured records; note any parsing/format inconsistencies you would need to clean up.

## Self-Assessment: Output Evaluation and Validation

Answer each question, then check the Answers cell below.


**Q1.** What is the defining characteristic of a hallucination, as opposed to a simple mistake?

**Q2.** A summary of an internal report cites a specific statistic that was never in your source material. What should you do before sending it?

**Q3.** Which categories of content should almost always receive human review before being published externally?

**Q4.** Why is adapting content for an audience considered a distinct skill from drafting it originally?

**Q5.** When should you choose a structured-data output format over inline prose?

## Answers


**A1.** A hallucination is a confidently and plausibly stated detail that is fabricated rather than sourced from real information.

**A2.** Flag it as unverified and check it against your actual source data (or remove it) before it is sent.

**A3.** Legal, financial, medical/compliance, and anything going to an external audience at scale.

**A4.** Because it requires re-framing the same underlying facts for different priorities, tone, and vocabulary, rather than generating new content.

**A5.** When the output will be consumed programmatically or imported into another tool (e.g., a spreadsheet or CRM), not just read once.

---
**Next**: [Domain 3: Product and Model Selection](./03-product-and-model-selection.ipynb)
**Back**: [README](./README.md)